# Analyze Train Metrics

Set of visualization tools to check if run is healthy. Requires training with `--log-metrics`.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
RUN_DIR = "../runs/d12m5"
assert os.path.exists(RUN_DIR), f"Run directory {RUN_DIR} does not exist."

In [ ]:
# List files
log_files = [fn for fn in os.listdir(RUN_DIR) if fn.startswith("train_log_") and fn.endswith(".jsonl")]
log_files = sorted(log_files)  # sort files to ensure rank order
print(log_files)

In [ ]:
# Load logs
log_objects = []
for log_file in log_files:
    with open(os.path.join(RUN_DIR, log_file), "r") as f:
        lines = f.readlines()
        log_objects.extend([json.loads(line) for line in lines])
        print(f"{log_file}: {len(lines)} lines")

In [ ]:
# First log object
for k, v in log_objects[0].items():
    print(f"{k}: {v}")

In [ ]:
# Filter 'train' events
log_objects = [obj for obj in log_objects if obj['event'] == 'train']
for k, v in log_objects[0].items():
    print(f"{k}: {v}")

In [ ]:
# Flatten metrics
metrics_all = []
for obj in log_objects:
    for m in obj['metrics']:
        metrics_all.append({'step': obj['step'], 'rank': obj['rank'], **m})

df = pd.DataFrame(metrics_all)
df["block"] = df["block"].astype("Int64")
print("----- Head -----")
display(df.head())
print("----- Summary -----")
display(df.describe())
print("----- NaNs per column -----")
display(df.isna().sum())
# df types
print("----- Data Types -----")
display(df.dtypes)
print("----- Unique Values -----")
print("Tensor:", sorted(df["tensor_name"].dropna().unique()))
print("Surface:", sorted(df["surface"].dropna().unique()))
print("Stat:", sorted(df["stat"].dropna().unique()))


## Post-Block Residual RMS

In [ ]:
# Prepare forward residuals for plotting
df_fwd_filtered = df[(df["tensor_name"] == "resid_post") & (df["surface"] == "fwd")].copy()  # filter fwd activations only
df_fwd_summed = df_fwd_filtered.groupby(["step", "block", "stat"], as_index=False)["value"].sum()  # sum over ranks
df_fwd_pivoted = df_fwd_summed.pivot(index=["step", "block"], columns="stat", values="value").reset_index()  # put sq_sum and num_el side by side
df_fwd_pivoted["rms"] = np.sqrt(df_fwd_pivoted["sq_sum"] / df_fwd_pivoted["num_el"])  # compute RMS
display(df_fwd_pivoted.head())

# Plot forward residual RMS by block
plt.figure(figsize=(12, 5))
for block_id, g in df_fwd_pivoted.groupby("block"):
    plt.plot(g["step"], g["rms"], label=f"layer {block_id}")
plt.title("Post-Block Residual RMS by Layer (d12)")
plt.xlabel("step")
plt.ylabel("RMS")
plt.legend(ncol=2, fontsize=8)
plt.show()

## Parm Gradient RMS

In [ ]:
core_block_tensors = [
    "attn.c_q.weight",
    "attn.c_k.weight",
    "attn.c_v.weight",
    "attn.c_proj.weight",
    "mlp.c_fc.weight",
    "mlp.c_proj.weight",
    # "value_embed.weight",  # excluded on purpose
    # "attn.ve_gate.weight"  # excluded on purpose
]

df_grad_filtered = df[(df['surface'] == 'grad') & (df["tensor_name"].isin(core_block_tensors))].copy()
df_grad_summed = df_grad_filtered.groupby(["step", "block", "stat"], as_index=False)["value"].sum()  # sum over ranks
df_grad_pivoted = df_grad_summed.pivot(index=["step", "block"], columns="stat", values="value").reset_index()  # put sq_sum and num_el side by side
df_grad_pivoted["rms"] = np.sqrt(df_grad_pivoted["sq_sum"] / df_grad_pivoted["num_el"])  # compute RMS
display(df_grad_pivoted.head())

# Plot gradient RMS by layer
plt.figure(figsize=(12, 5))
for block_id, g in df_grad_pivoted.groupby("block"):
    plt.plot(g["step"], g["rms"], label=f"layer {block_id}")
plt.title("Param Gradient RMS by Layer (d12, attn+mlp)")
plt.xlabel("step")
plt.ylabel("RMS")
plt.yscale("log")
plt.legend(ncol=2, fontsize=8)
plt.show()

## Param Update Ratio

In [ ]:
core_block_tensors = [
    "attn.c_q.weight",
    "attn.c_k.weight",
    "attn.c_v.weight",
    "attn.c_proj.weight",
    "mlp.c_fc.weight",
    "mlp.c_proj.weight",
    # "value_embed.weight",  # excluded on purpose
    # "attn.ve_gate.weight"  # excluded on purpose
]

df_update_filtered = df[(df['surface'].isin(['update', 'params'])) & (df['stat'] == 'sq_sum') & (df["tensor_name"].isin(core_block_tensors))].copy()
df_update_summed = df_update_filtered.groupby(["step", "block", "surface"], as_index=False)["value"].sum()  # sum over ranks
df_update_pivoted = df_update_summed.pivot(index=["step", "block"], columns="surface", values="value").reset_index()  # put sq_sum and num_el side by side
df_update_pivoted["update_ratio"] = np.sqrt(df_update_pivoted["update"] / df_update_pivoted["params"])  # compute update ratio
display(df_update_pivoted.head())

# Plot gradient RMS by block
plt.figure(figsize=(12, 5))
for block_id, g in df_update_pivoted.groupby("block"):
    plt.plot(g["step"], g["update_ratio"], label=f"layer {block_id}")
plt.title("Parameter Update Ratio by Layer (d12, attn+mlp)")
plt.xlabel("step")
plt.ylabel("Update Ratio")
plt.yscale("log")
plt.legend(ncol=2, fontsize=8)
plt.show()